# Question 1 — Stage 4: Clean and preprocess the data

**Group number:** 4
**Student 1:** Niyonsenga Felicien
**Student 2:** Leonce
**Question:** Q1 — Maize yield in Rwanda  
**Stage:** 4

Cleaning is performed in code. The raw CSV remains unchanged. Missing-value imputers and scalers are fitted using the training data only. The target `yield_t_ha` is separated from predictors.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
sns.set_theme()
CWD = Path.cwd()
raw_candidates = [
    CWD / "data" / "raw" / "maize_yield_rwanda_rw.csv",
    CWD / "q1_maize_yield" / "data" / "raw" / "maize_yield_rwanda_rw.csv",
    CWD.parent / "data" / "raw" / "maize_yield_rwanda_rw.csv",
]
RAW_PATH = next((p for p in raw_candidates if p.exists()), None)
if RAW_PATH is None:
    raise FileNotFoundError("Could not find data/raw/maize_yield_rwanda_rw.csv. Open the notebook from the extracted project.")
PROJECT = RAW_PATH.parents[2]
INTERIM = PROJECT / "data" / "interim"
PROCESSED = PROJECT / "data" / "processed"
INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(RAW_PATH)

print("Before cleaning:", df.shape)
print("Missing values before:")
display(df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())

Before cleaning: (1426, 20)
Missing values before:


record_id                0
year                     0
season                   0
province                 0
district                 0
altitude_m               0
rainfall_mm             11
avg_temp_c               0
soil_ph                 14
soil_type                0
seed_variety             0
farming_practice         0
npk_kg_ha                0
urea_kg_ha               0
manure_t_ha              0
plant_density_per_ha     0
farm_area_ha             0
irrigation               0
pest_incidence_index    14
yield_t_ha               0
dtype: int64

Duplicate rows: 0


## Cleaning decisions
- Remove exact duplicate rows if present.
- Standardise whitespace in text categories.
- Convert the year to integer (already numeric in this file).
- Treat `record_id` as an identifier, not a predictor.
- Keep the target `yield_t_ha` separate.
- Do not automatically remove IQR outliers: review them first because extreme but valid farm conditions may exist.
- Impute missing numeric values with medians and categorical values with most frequent values inside a scikit-learn preprocessing pipeline fitted on training data only.

In [3]:
clean = df.copy()
before_rows = len(clean)
clean = clean.drop_duplicates().copy()
for col in clean.select_dtypes(include="object").columns:
    clean[col] = clean[col].astype("string").str.strip()
clean["year"] = pd.to_numeric(clean["year"], errors="coerce").astype("Int64")
print("Rows removed as exact duplicates:", before_rows-len(clean))
print("After basic cleaning:", clean.shape)
display(clean.head())

Rows removed as exact duplicates: 0
After basic cleaning: (1426, 20)


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_13928\610777642.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in clean.select_dtypes(include="object").columns:


,record_id,year,season,province,district,altitude_m,rainfall_mm,avg_temp_c,soil_ph,soil_type,seed_variety,farming_practice,npk_kg_ha,urea_kg_ha,manure_t_ha,plant_density_per_ha,farm_area_ha,irrigation,pest_incidence_index,yield_t_ha
0,RW-MZ-00001,2015,A,Eastern,Bugesera,1349,NaN,22.9,NaN,Clay,RHM104,Improved,117.8,57.4,1.41,61803,0.63,No,NaN,2.71
1,RW-MZ-00002,2015,A,Eastern,Bugesera,1427,489.8,22.1,6.02,Volcanic,RHM104,Improved,38.1,62.7,3.69,52919,0.17,No,0.339,4.09
2,RW-MZ-00003,2015,A,Northern,Burera,2033,722.6,16.3,5.73,Clay loam,Local,Improved,121.1,50.7,5.18,54406,0.22,No,0.049,2.58
3,RW-MZ-00004,2015,A,Northern,Burera,2026,794.4,16.4,5.20,Clay loam,ZM621,Improved,93.7,60.4,1.68,48119,0.48,No,0.171,2.45
4,RW-MZ-00005,2015,A,Northern,Gakenke,1974,644.2,18.3,5.92,Loam,Local,Improved,84.1,70.4,0.36,56787,1.16,No,0.001,2.09


## Train / validation / test split
We use a fixed random seed and a 70% / 15% / 15% split. Because the data spans years, discuss with your lecturer whether a chronological split is more appropriate for the intended real-world forecasting use. This random split is the reproducible baseline requested by the assignment example.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET = "yield_t_ha"
ID_COL = "record_id"
model_data = clean.dropna(subset=[TARGET]).copy()
X = model_data.drop(columns=[TARGET, ID_COL])
y = model_data[TARGET].astype(float)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)
print("Train:", X_train.shape, "Validation:", X_valid.shape, "Test:", X_test.shape)

Train: (998, 18) Validation: (214, 18) Test: (214, 18)


## Reusable preprocessing pipeline
Numerical columns are median-imputed and standardised. Categorical columns are mode-imputed and one-hot encoded. The transformer is fitted only on the training set; validation and test are transformed using the already-fitted transformer.

In [5]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_ready = preprocessor.fit_transform(X_train)
X_valid_ready = preprocessor.transform(X_valid)
X_test_ready = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

X_train_ready = pd.DataFrame(X_train_ready, columns=feature_names, index=X_train.index)
X_valid_ready = pd.DataFrame(X_valid_ready, columns=feature_names, index=X_valid.index)
X_test_ready = pd.DataFrame(X_test_ready, columns=feature_names, index=X_test.index)

print("Transformed shapes:", X_train_ready.shape, X_valid_ready.shape, X_test_ready.shape)
display(X_train_ready.head())

Transformed shapes: (998, 64) (214, 64) (214, 64)


,num__year,num__altitude_m,num__rainfall_mm,num__avg_temp_c,num__soil_ph,num__npk_kg_ha,num__urea_kg_ha,num__manure_t_ha,num__plant_density_per_ha,num__farm_area_ha,...,cat__seed_variety_Local,cat__seed_variety_PAN691,cat__seed_variety_RHM104,cat__seed_variety_WH505,cat__seed_variety_ZM607,cat__seed_variety_ZM621,cat__farming_practice_Improved,cat__farming_practice_Traditional,cat__irrigation_No,cat__irrigation_Yes
390,-0.860897,0.772074,0.014741,-0.430069,-1.733868,-1.391477,-0.216704,-0.209508,-1.987868,-1.338987,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
573,-0.165490,0.961328,1.530772,-0.976832,0.254993,-0.854603,0.587311,-0.953178,0.576370,5.997153,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
807,0.182213,-1.647674,-1.247534,2.060740,0.735063,-0.704877,-0.261910,-0.187419,-1.918933,-0.539036,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1228,1.225323,0.141227,0.166995,-0.551572,-0.247938,-1.391477,-1.204771,0.924404,-0.806228,-0.519525,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
585,-0.165490,0.055612,0.191720,0.116694,1.146551,-0.587235,-0.736569,0.210187,-1.619497,0.768202,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0


## Save clean data and processed splits
The `*_features.csv` files contain transformed predictors plus the target. The original `record_id` is excluded from modelling but saved separately in the cleaned descriptive dataset for traceability.

In [6]:
clean.to_csv(INTERIM/"maize_clean.csv", index=False)
train_out = X_train_ready.copy(); train_out[TARGET] = y_train
valid_out = X_valid_ready.copy(); valid_out[TARGET] = y_valid
test_out = X_test_ready.copy(); test_out[TARGET] = y_test
train_out.to_csv(PROCESSED/"train_features.csv", index=False)
valid_out.to_csv(PROCESSED/"validation_features.csv", index=False)
test_out.to_csv(PROCESSED/"test_features.csv", index=False)
print("Saved cleaned data and train/validation/test feature files.")

Saved cleaned data and train/validation/test feature files.


## Cleaning Log

In [7]:
cleaning_log = pd.DataFrame([
    ("Exact duplicate removal","All columns",int(before_rows-len(clean)),"Drop exact duplicate rows","Avoid duplicate weighting"),
    ("Text trimming","Object/string columns",int(len(clean)),"Strip leading/trailing whitespace","Standardise category labels"),
    ("Missing numeric values","Numeric predictor columns",int(clean[numeric_features].isna().sum().sum()),"Median imputation within training pipeline","Robust to skew; fit on train only"),
    ("Missing categorical values","Categorical predictor columns",int(clean[categorical_features].isna().sum().sum()),"Most-frequent imputation within training pipeline","Preserve records"),
    ("Identifier exclusion","record_id",int(clean[ID_COL].notna().sum()),"Exclude from X; retain in cleaned file","Avoid learning arbitrary ID patterns"),
    ("Target separation","yield_t_ha",int(clean[TARGET].notna().sum()),"Keep as y, not in X","Prevent target leakage"),
], columns=["operation","column","records_or_values_affected","action","reason"])
display(cleaning_log)
cleaning_log.to_csv(PROJECT/"docs"/"cleaning_log.csv", index=False)

,operation,column,records_or_values_affected,action,reason
0,Exact duplicate removal,All columns,0,Drop exact duplicate rows,Avoid duplicate weighting
1,Text trimming,Object/string columns,1426,Strip leading/trailing whitespace,Standardise category labels
2,Missing numeric values,Numeric predictor columns,39,Median imputation within training pipeline,Robust to skew; fit on train only
3,Missing categorical values,Categorical predictor columns,0,Most-frequent imputation within training pipeline,Preserve records
4,Identifier exclusion,record_id,1426,Exclude from X; retain in cleaned file,Avoid learning arbitrary ID patterns
5,Target separation,yield_t_ha,1426,"Keep as y, not in X",Prevent target leakage
